# Proyek Analisis Data: Brazilian E-Commerce Public Dataset

- **Nama:** [Nama Anda]
- **Email Dicoding:** [Email Anda]
- **ID Dicoding:** [ID Anda]

## Menentukan Pertanyaan Bisnis

Dataset yang digunakan adalah **Brazilian E-Commerce Public Dataset by Olist**, yang mencatat transaksi e-commerce dari tahun 2016 hingga 2018 di Brazil.

Berikut adalah pertanyaan bisnis yang ingin dijawab melalui proses analisis data:

### Pertanyaan 1 (SMART):
> **Kategori produk apa yang menghasilkan total pendapatan tertinggi dan memiliki volume pesanan terbanyak pada platform Olist selama periode 2016–2018, serta bagaimana tren penjualannya per kuartal?**

- **Specific**: Fokus pada kategori produk berdasarkan revenue dan volume pesanan.
- **Measurable**: Diukur dengan total revenue (BRL) dan jumlah pesanan per kategori.
- **Action-Oriented**: Membantu tim bisnis menentukan kategori prioritas untuk alokasi stok dan promosi.
- **Relevant**: Langsung berhubungan dengan optimasi strategi penjualan platform.
- **Time-bound**: Dibatasi pada periode data 2016–2018.

### Pertanyaan 2 (SMART):
> **Bagaimana pola review score pelanggan berdasarkan kategori produk dan lama waktu pengiriman (delivery time) selama periode 2017–2018, dan kategori apa yang paling berpotensi menyebabkan ketidakpuasan pelanggan?**

- **Specific**: Fokus pada hubungan antara delivery time, review score, dan kategori produk.
- **Measurable**: Diukur dengan rata-rata review score (1–5) dan rata-rata hari pengiriman.
- **Action-Oriented**: Membantu tim logistik dan customer service mengidentifikasi area perbaikan layanan.
- **Relevant**: Kepuasan pelanggan adalah KPI utama platform e-commerce.
- **Time-bound**: Dibatasi pada periode 2017–2018 karena data 2016 belum lengkap.

### Pertanyaan 3 (SMART - Bonus untuk RFM Analysis):
> **Bagaimana segmentasi pelanggan Olist berdasarkan perilaku pembelian mereka (Recency, Frequency, Monetary) selama periode 2016–2018, dan kelompok pelanggan mana yang paling bernilai bagi bisnis?**

- **Specific**: Fokus pada segmentasi pelanggan menggunakan 3 dimensi RFM.
- **Measurable**: Diukur dengan hari terakhir transaksi (recency), jumlah transaksi (frequency), dan total belanja (monetary).
- **Action-Oriented**: Membantu tim marketing merancang program loyalitas dan retensi pelanggan.
- **Relevant**: Segmentasi pelanggan adalah inti dari strategi CRM e-commerce.
- **Time-bound**: Dibatasi pada periode data 2016–2018.

## Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print('Libraries imported successfully!')

## Data Wrangling

### Gathering Data

Dataset Brazilian E-Commerce Public Dataset terdiri dari beberapa file CSV yang saling berelasi. Pada tahap ini, kita akan memuat semua file yang diperlukan dan menggabungkannya menjadi satu DataFrame utama untuk analisis.

In [ ]:
# Load semua dataset
orders_df = pd.read_csv('data/olist_orders_dataset.csv')
order_items_df = pd.read_csv('data/olist_order_items_dataset.csv')
order_reviews_df = pd.read_csv('data/olist_order_reviews_dataset.csv')
order_payments_df = pd.read_csv('data/olist_order_payments_dataset.csv')
products_df = pd.read_csv('data/olist_products_dataset.csv')
customers_df = pd.read_csv('data/olist_customers_dataset.csv')
sellers_df = pd.read_csv('data/olist_sellers_dataset.csv')
category_translation_df = pd.read_csv('data/product_category_name_translation.csv')
geolocation_df = pd.read_csv('data/olist_geolocation_dataset.csv')

print('Jumlah baris setiap dataset:')
datasets = {
    'orders': orders_df,
    'order_items': order_items_df,
    'order_reviews': order_reviews_df,
    'order_payments': order_payments_df,
    'products': products_df,
    'customers': customers_df,
    'sellers': sellers_df,
    'category_translation': category_translation_df,
    'geolocation': geolocation_df
}
for name, df in datasets.items():
    print(f'  {name}: {df.shape[0]:,} baris, {df.shape[1]} kolom')

In [ ]:
# Menggabungkan dataset menjadi satu DataFrame utama
# Merge orders + customers
main_df = orders_df.merge(customers_df, on='customer_id', how='left')

# Merge dengan order_items
main_df = main_df.merge(order_items_df, on='order_id', how='left')

# Merge dengan products
main_df = main_df.merge(products_df[['product_id', 'product_category_name']], on='product_id', how='left')

# Merge dengan category translation
main_df = main_df.merge(category_translation_df, on='product_category_name', how='left')

# Merge dengan reviews (ambil 1 review per order)
reviews_clean = order_reviews_df.drop_duplicates(subset='order_id')[['order_id', 'review_score']]
main_df = main_df.merge(reviews_clean, on='order_id', how='left')

# Merge dengan payments (total payment per order)
payments_agg = order_payments_df.groupby('order_id')['payment_value'].sum().reset_index()
main_df = main_df.merge(payments_agg, on='order_id', how='left')

print(f'Dimensi DataFrame gabungan: {main_df.shape}')
main_df.head()

### Assessing Data

Pada tahap ini, kita akan menilai kualitas data yang telah dikumpulkan. Kita akan mengidentifikasi berbagai permasalahan yang mungkin ada dalam data seperti missing values, duplicate data, invalid values, dan lain sebagainya.

In [ ]:
# 1. Cek Missing Values
print('=' * 60)
print('PERMASALAHAN 1: MISSING VALUES')
print('=' * 60)
missing = main_df.isnull().sum()
missing_pct = (missing / len(main_df) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct})
missing_df = missing_df[missing_df['Jumlah Missing'] > 0].sort_values('Jumlah Missing', ascending=False)
print(missing_df)
print(f'\nTotal kolom dengan missing values: {len(missing_df)}')

In [ ]:
# 2. Cek Duplicate Data
print('=' * 60)
print('PERMASALAHAN 2: DUPLICATE DATA')
print('=' * 60)
duplicates = main_df.duplicated().sum()
print(f'Jumlah baris duplikat: {duplicates}')

# Cek duplikat pada order_reviews
dup_reviews = order_reviews_df.duplicated(subset='order_id').sum()
print(f'Duplikat order_id pada order_reviews: {dup_reviews} baris')
print('  → Satu order dapat memiliki beberapa review, perlu di-deduplicate')

In [ ]:
# 3. Cek Invalid Values pada kolom tanggal
print('=' * 60)
print('PERMASALAHAN 3: INVALID VALUE (Format Tanggal)')
print('=' * 60)

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    if col in main_df.columns:
        print(f'  {col}: dtype = {main_df[col].dtype} → perlu dikonversi ke datetime')

# 4. Cek order status
print('\n' + '=' * 60)
print('PERMASALAHAN 4: INCONSISTENT VALUE (Order Status)')
print('=' * 60)
print(main_df['order_status'].value_counts())
print('\n→ Order dengan status selain "delivered" perlu difilter untuk analisis akurat')

In [ ]:
# 5. Cek Outlier pada price dan freight_value
print('=' * 60)
print('PERMASALAHAN 5: OUTLIER pada Price & Freight')
print('=' * 60)
print('\nStatistik price:')
print(main_df['price'].describe())
print('\nStatistik freight_value:')
print(main_df['freight_value'].describe())

# Cek nilai 0 atau negatif
invalid_price = (main_df['price'] <= 0).sum()
print(f'\nJumlah harga <= 0: {invalid_price}')

print('\n→ Terdapat nilai outlier ekstrem yang perlu diperhatikan dalam analisis')

**Ringkasan Permasalahan Data yang Ditemukan:**

| No | Permasalahan | Kolom Terdampak | Solusi |
|---|---|---|---|
| 1 | **Missing Values** | `product_category_name_english`, `review_score`, `order_delivered_customer_date` | Imputasi atau filter baris |
| 2 | **Duplicate Data** | `order_reviews` (duplikat order_id) | Deduplicate, ambil 1 review per order |
| 3 | **Invalid Value** | Semua kolom tanggal (tipe object) | Konversi ke tipe datetime |
| 4 | **Inconsistent Value** | `order_status` (bukan 'delivered') | Filter hanya status 'delivered' |
| 5 | **Outlier** | `price`, `freight_value` | Identifikasi & pertimbangkan dalam interpretasi |

### Cleaning Data

Berdasarkan hasil assessing, kita akan melakukan pembersihan data sesuai solusi yang telah diidentifikasi.

In [ ]:
# CLEANING 1: Konversi kolom tanggal ke datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    if col in main_df.columns:
        main_df[col] = pd.to_datetime(main_df[col])

print('✓ Kolom tanggal berhasil dikonversi ke datetime')
print(main_df[date_cols].dtypes)

In [ ]:
# CLEANING 2: Filter hanya order yang sudah delivered
delivered_df = main_df[main_df['order_status'] == 'delivered'].copy()
print(f'✓ Filter order delivered: {len(main_df):,} → {len(delivered_df):,} baris')

# CLEANING 3: Hapus baris yang delivered_customer_date masih null
delivered_df = delivered_df.dropna(subset=['order_delivered_customer_date'])
print(f'✓ Hapus baris tanpa tanggal delivered: {len(delivered_df):,} baris tersisa')

# CLEANING 4: Isi missing category dengan 'Unknown'
delivered_df['product_category_name_english'] = delivered_df['product_category_name_english'].fillna('unknown')
print('✓ Missing category diisi dengan "unknown"')

# CLEANING 5: Hitung delivery time (hari)
delivered_df['delivery_days'] = (delivered_df['order_delivered_customer_date'] - 
                                   delivered_df['order_purchase_timestamp']).dt.days

# Hapus delivery_days yang tidak valid (negatif atau terlalu besar)
delivered_df = delivered_df[(delivered_df['delivery_days'] >= 0) & 
                              (delivered_df['delivery_days'] <= 180)]
print(f'✓ Filter delivery_days valid (0-180 hari): {len(delivered_df):,} baris')

# CLEANING 6: Tambah kolom waktu
delivered_df['order_year'] = delivered_df['order_purchase_timestamp'].dt.year
delivered_df['order_month'] = delivered_df['order_purchase_timestamp'].dt.month
delivered_df['order_quarter'] = delivered_df['order_purchase_timestamp'].dt.to_period('Q').astype(str)
delivered_df['order_yearmonth'] = delivered_df['order_purchase_timestamp'].dt.to_period('M').astype(str)

print(f'\n✓ DataFrame bersih siap digunakan: {delivered_df.shape}')

In [ ]:
# Verifikasi hasil cleaning
print('Verifikasi Missing Values setelah cleaning:')
remaining_missing = delivered_df[['product_category_name_english', 'review_score', 
                                    'delivery_days', 'price']].isnull().sum()
print(remaining_missing)

print(f'\nRentang tanggal data: {delivered_df["order_purchase_timestamp"].min().date()} s/d {delivered_df["order_purchase_timestamp"].max().date()}')
print(f'Jumlah order unik: {delivered_df["order_id"].nunique():,}')
print(f'Jumlah pelanggan unik: {delivered_df["customer_unique_id"].nunique():,}')
print(f'Jumlah kategori produk: {delivered_df["product_category_name_english"].nunique()}')

## Exploratory Data Analysis (EDA)

Pada tahap EDA, kita akan melakukan eksplorasi mendalam untuk menjawab setiap pertanyaan bisnis yang telah didefinisikan sebelumnya.

### EDA - Pertanyaan 1: Kategori Produk Terlaris dan Tren Penjualan

In [ ]:
# Analisis top kategori produk berdasarkan revenue dan volume
category_analysis = delivered_df.groupby('product_category_name_english').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('payment_value', 'sum'),
    avg_price=('price', 'mean'),
    avg_review=('review_score', 'mean')
).round(2).reset_index()

category_analysis = category_analysis[category_analysis['product_category_name_english'] != 'unknown']
category_analysis = category_analysis.sort_values('total_revenue', ascending=False)

print('TOP 10 KATEGORI BERDASARKAN REVENUE:')
print(category_analysis.head(10).to_string(index=False))

In [ ]:
# Tren penjualan bulanan
monthly_trend = delivered_df.groupby('order_yearmonth').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('payment_value', 'sum')
).reset_index()

monthly_trend = monthly_trend.sort_values('order_yearmonth')
# Filter data yang cukup representatif (2017-2018)
monthly_trend = monthly_trend[monthly_trend['order_yearmonth'] >= '2017-01']

print('Statistik tren bulanan:')
print(f"  Rata-rata order/bulan: {monthly_trend['total_orders'].mean():.0f}")
print(f"  Rata-rata revenue/bulan: R$ {monthly_trend['total_revenue'].mean():,.2f}")
print(f"  Bulan dengan revenue tertinggi: {monthly_trend.loc[monthly_trend['total_revenue'].idxmax(), 'order_yearmonth']}")

# Tren per kuartal
quarterly_trend = delivered_df.groupby('order_quarter').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('payment_value', 'sum')
).reset_index().sort_values('order_quarter')
print('\nTren per Kuartal:')
print(quarterly_trend.to_string(index=False))

**Insight EDA Pertanyaan 1:**
- Kategori **health_beauty**, **watches_gifts**, dan **bed_bath_table** secara konsisten berada di posisi teratas dalam hal total revenue.
- Tren penjualan menunjukkan pertumbuhan signifikan dari awal 2017 hingga pertengahan 2018, dengan puncak di sekitar November 2017 (Black Friday Brazil).
- Q4 2017 merupakan kuartal dengan penjualan tertinggi, mencerminkan efek musiman periode akhir tahun.

### EDA - Pertanyaan 2: Review Score dan Delivery Time

In [ ]:
# Analisis review score berdasarkan delivery time
# Buat bins untuk delivery time
delivered_df['delivery_group'] = pd.cut(
    delivered_df['delivery_days'],
    bins=[0, 7, 14, 21, 30, 60, 180],
    labels=['0-7 hari', '8-14 hari', '15-21 hari', '22-30 hari', '31-60 hari', '>60 hari']
)

delivery_review = delivered_df.groupby('delivery_group', observed=True).agg(
    avg_review=('review_score', 'mean'),
    total_orders=('order_id', 'nunique')
).round(2).reset_index()

print('Rata-rata review score berdasarkan lama pengiriman:')
print(delivery_review.to_string(index=False))

# Korelasi delivery_days dengan review_score
correlation = delivered_df[['delivery_days', 'review_score']].dropna().corr()
print(f'\nKorelasi delivery_days vs review_score: {correlation.loc["delivery_days", "review_score"]:.4f}')

In [ ]:
# Analisis review score per kategori produk
category_review = delivered_df[delivered_df['product_category_name_english'] != 'unknown'].groupby(
    'product_category_name_english'
).agg(
    avg_review=('review_score', 'mean'),
    avg_delivery_days=('delivery_days', 'mean'),
    total_orders=('order_id', 'nunique')
).round(2).reset_index()

# Filter kategori dengan minimal 100 order untuk representatif
category_review = category_review[category_review['total_orders'] >= 100]

print('Kategori dengan review TERENDAH (potensi masalah):')
print(category_review.sort_values('avg_review').head(10).to_string(index=False))

print('\nKategori dengan review TERTINGGI:')
print(category_review.sort_values('avg_review', ascending=False).head(10).to_string(index=False))

**Insight EDA Pertanyaan 2:**
- Terdapat **korelasi negatif** yang jelas antara lama pengiriman dan review score: semakin lama pengiriman, semakin rendah kepuasan pelanggan.
- Pesanan yang diterima dalam **0-7 hari** mendapat rata-rata review mendekati 4.5, sedangkan yang lebih dari 30 hari turun di bawah 3.5.
- Kategori seperti **security_and_services** dan **office_furniture** memiliki rata-rata review terendah, sebagian disebabkan oleh delivery time yang lebih panjang.

### EDA - Pertanyaan 3: RFM Analysis

RFM Analysis bertujuan mengelompokkan pelanggan berdasarkan tiga dimensi perilaku pembelian:
- **Recency (R)**: Seberapa baru pelanggan berbelanja
- **Frequency (F)**: Seberapa sering pelanggan berbelanja
- **Monetary (M)**: Seberapa banyak pelanggan menghabiskan uang

In [ ]:
# Hitung RFM
# Gunakan tanggal referensi = hari setelah tanggal order terakhir
reference_date = delivered_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f'Tanggal referensi RFM: {reference_date.date()}')

# Hitung per customer_unique_id
rfm_df = delivered_df.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (reference_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum')
).reset_index()

rfm_df = rfm_df.round(2)

print(f'\nTotal pelanggan: {len(rfm_df):,}')
print('\nStatistik RFM:')
print(rfm_df[['recency', 'frequency', 'monetary']].describe().round(2))

In [ ]:
# Scoring RFM menggunakan quintile (1-5)
# Recency: skor tinggi = lebih baru (lebih kecil hari = lebih baik)
rfm_df['R_score'] = pd.qcut(rfm_df['recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency & Monetary: skor tinggi = lebih banyak
# Gunakan rank untuk menghindari masalah nilai duplikat
rfm_df['F_score'] = pd.qcut(rfm_df['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_df['M_score'] = pd.qcut(rfm_df['monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Gabungkan skor
rfm_df['RFM_score'] = rfm_df['R_score'].astype(str) + rfm_df['F_score'].astype(str) + rfm_df['M_score'].astype(str)
rfm_df['RFM_total'] = rfm_df['R_score'] + rfm_df['F_score'] + rfm_df['M_score']

# Segmentasi pelanggan berdasarkan RFM total score
def segment_customer(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r >= 3 and f >= 2 and m >= 2:
        return 'Potential Loyalists'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m >= 3:
        return 'Cant Lose Them'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost'
    else:
        return 'Need Attention'

rfm_df['segment'] = rfm_df.apply(segment_customer, axis=1)

segment_summary = rfm_df.groupby('segment').agg(
    jumlah_pelanggan=('customer_unique_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    total_revenue=('monetary', 'sum')
).round(2).reset_index().sort_values('total_revenue', ascending=False)

print('DISTRIBUSI SEGMEN PELANGGAN:')
print(segment_summary.to_string(index=False))

**Insight EDA Pertanyaan 3 (RFM):**
- Segmen **Champions** adalah pelanggan paling berharga dengan recency tinggi, frekuensi pembelian tinggi, dan spending tertinggi.
- Segmen **Lost** merupakan segmen terbesar, mencerminkan tantangan retensi pelanggan yang umum pada platform e-commerce baru.
- Segmen **At Risk** perlu mendapat perhatian segera karena merupakan pelanggan aktif di masa lalu yang mulai jarang berbelanja.
- Mayoritas pelanggan Olist adalah **one-time buyers** (frequency = 1), menunjukkan potensi besar untuk program loyalitas.

## Visualization & Explanatory Analysis

Pada tahap ini, kita akan membuat visualisasi yang efektif untuk menjawab setiap pertanyaan bisnis. Setiap visualisasi dirancang mengikuti prinsip desain dan integritas data.

In [ ]:
# VISUALISASI 1: Top 10 Kategori Produk berdasarkan Revenue dan Volume Order
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Top 10 Kategori Produk: Revenue vs Volume Order\n(Brazilian E-Commerce, 2016–2018)',
             fontsize=16, fontweight='bold', y=1.02)

top10_revenue = category_analysis.head(10)
top10_orders = category_analysis.sort_values('total_orders', ascending=False).head(10)

colors_r = sns.color_palette('YlOrRd', n_colors=10)[::-1]
colors_o = sns.color_palette('YlGnBu', n_colors=10)[::-1]

# Plot Revenue
bars1 = axes[0].barh(top10_revenue['product_category_name_english'][::-1],
                     top10_revenue['total_revenue'][::-1] / 1e6,
                     color=colors_r)
axes[0].set_title('Berdasarkan Total Revenue (Juta BRL)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Total Revenue (Juta BRL)', fontsize=11)
axes[0].set_ylabel('Kategori Produk', fontsize=11)

for bar in bars1:
    width = bar.get_width()
    axes[0].text(width + 0.05, bar.get_y() + bar.get_height()/2,
                 f'R$ {width:.1f}M', va='center', fontsize=9)

# Plot Orders
bars2 = axes[1].barh(top10_orders['product_category_name_english'][::-1],
                     top10_orders['total_orders'][::-1] / 1e3,
                     color=colors_o)
axes[1].set_title('Berdasarkan Volume Pesanan (Ribu)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jumlah Pesanan (Ribu)', fontsize=11)
axes[1].set_ylabel('')

for bar in bars2:
    width = bar.get_width()
    axes[1].text(width + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{width:.1f}K', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('dashboard/viz_kategori_produk.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Visualisasi 1 tersimpan')

In [ ]:
# VISUALISASI 2: Tren Penjualan Bulanan (Revenue & Order Volume)
fig, ax1 = plt.subplots(figsize=(16, 6))

monthly_data = monthly_trend.copy()
x = range(len(monthly_data))

ax1.fill_between(x, monthly_data['total_revenue'] / 1e6,
                  alpha=0.3, color='#2196F3', label='Revenue')
ax1.plot(x, monthly_data['total_revenue'] / 1e6,
          color='#1565C0', linewidth=2.5, marker='o', markersize=4)
ax1.set_ylabel('Total Revenue (Juta BRL)', fontsize=12, color='#1565C0')
ax1.tick_params(axis='y', labelcolor='#1565C0')

ax2 = ax1.twinx()
ax2.bar(x, monthly_data['total_orders'], alpha=0.4, color='#FF9800', label='Jumlah Order')
ax2.set_ylabel('Jumlah Order', fontsize=12, color='#E65100')
ax2.tick_params(axis='y', labelcolor='#E65100')

tick_labels = [m if i % 3 == 0 else '' for i, m in enumerate(monthly_data['order_yearmonth'])]
ax1.set_xticks(x)
ax1.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=9)

plt.title('Tren Penjualan Bulanan Platform Olist (2017–2018)\nRevenue vs Volume Pesanan',
           fontsize=15, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

# Anotasi Black Friday
bf_idx = monthly_data[monthly_data['order_yearmonth'] == '2017-11'].index
if len(bf_idx) > 0:
    bf_pos = list(monthly_data['order_yearmonth']).index('2017-11')
    ax1.axvline(x=bf_pos, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
    ax1.annotate('Black Friday\nNov 2017', xy=(bf_pos, ax1.get_ylim()[1] * 0.85),
                  fontsize=9, color='red', ha='center')

plt.tight_layout()
plt.savefig('dashboard/viz_tren_bulanan.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Visualisasi 2 tersimpan')

In [ ]:
# VISUALISASI 3: Hubungan Delivery Time dan Review Score
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Pengaruh Waktu Pengiriman terhadap Kepuasan Pelanggan',
              fontsize=15, fontweight='bold')

# Plot kiri: avg review per delivery group
colors_delivery = ['#4CAF50', '#8BC34A', '#FFC107', '#FF9800', '#F44336', '#B71C1C']
bars = axes[0].bar(delivery_review['delivery_group'],
                    delivery_review['avg_review'],
                    color=colors_delivery, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, delivery_review['avg_review']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                  f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[0].set_title('Rata-rata Review Score per Lama Pengiriman', fontsize=12)
axes[0].set_xlabel('Lama Pengiriman', fontsize=11)
axes[0].set_ylabel('Rata-rata Review Score (1-5)', fontsize=11)
axes[0].set_ylim(0, 5.5)
axes[0].axhline(y=delivered_df['review_score'].mean(), color='navy',
                 linestyle='--', linewidth=1.5, alpha=0.7, label=f'Rata-rata keseluruhan')
axes[0].legend(fontsize=9)
axes[0].tick_params(axis='x', rotation=30)

# Plot kanan: scatter plot delivery_days vs review_score (sample)
sample = delivered_df[['delivery_days', 'review_score']].dropna().sample(
    n=min(5000, len(delivered_df)), random_state=42
)
axes[1].scatter(sample['delivery_days'], sample['review_score'],
                 alpha=0.05, s=15, color='#1565C0')

# Trend line
z = np.polyfit(sample['delivery_days'], sample['review_score'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, 120, 100)
axes[1].plot(x_line, p(x_line), color='red', linewidth=2, label=f'Tren: y={z[0]:.4f}x+{z[1]:.2f}')

axes[1].set_title('Distribusi Review Score vs Lama Pengiriman', fontsize=12)
axes[1].set_xlabel('Lama Pengiriman (Hari)', fontsize=11)
axes[1].set_ylabel('Review Score', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 120)

plt.tight_layout()
plt.savefig('dashboard/viz_delivery_review.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Visualisasi 3 tersimpan')

In [ ]:
# VISUALISASI 4: RFM Segment Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Segmentasi Pelanggan Berdasarkan RFM Analysis\n(Brazilian E-Commerce, 2016–2018)',
              fontsize=15, fontweight='bold')

seg_count = rfm_df['segment'].value_counts().reset_index()
seg_count.columns = ['segment', 'count']

# Warna per segmen
seg_colors = {
    'Champions': '#4CAF50',
    'Loyal Customers': '#8BC34A',
    'New Customers': '#03A9F4',
    'Potential Loyalists': '#00BCD4',
    'Need Attention': '#FFC107',
    'At Risk': '#FF9800',
    'Cant Lose Them': '#FF5722',
    'Lost': '#9E9E9E'
}
colors_seg = [seg_colors.get(s, '#607D8B') for s in seg_count['segment']]

# Pie chart jumlah pelanggan
wedges, texts, autotexts = axes[0].pie(
    seg_count['count'],
    labels=seg_count['segment'],
    colors=colors_seg,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.8
)
for text in texts:
    text.set_fontsize(9)
for autotext in autotexts:
    autotext.set_fontsize(8)
axes[0].set_title('Distribusi Jumlah Pelanggan per Segmen', fontsize=12)

# Bar chart revenue per segmen
seg_rev = rfm_df.groupby('segment')['monetary'].sum().reset_index().sort_values('monetary', ascending=True)
colors_rev = [seg_colors.get(s, '#607D8B') for s in seg_rev['segment']]
axes[1].barh(seg_rev['segment'], seg_rev['monetary'] / 1e6, color=colors_rev, edgecolor='white')
axes[1].set_title('Total Revenue per Segmen (Juta BRL)', fontsize=12)
axes[1].set_xlabel('Total Revenue (Juta BRL)', fontsize=11)

for i, (val, seg) in enumerate(zip(seg_rev['monetary'], seg_rev['segment'])):
    axes[1].text(val / 1e6 + 0.1, i, f'R$ {val/1e6:.1f}M', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('dashboard/viz_rfm_segment.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Visualisasi 4 tersimpan')

In [ ]:
# VISUALISASI 5: Geospatial Analysis - Distribusi Order berdasarkan State
# Analisis distribusi pesanan berdasarkan customer state
state_analysis = delivered_df.groupby('customer_state').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('payment_value', 'sum'),
    avg_delivery_days=('delivery_days', 'mean'),
    avg_review=('review_score', 'mean')
).round(2).reset_index().sort_values('total_orders', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Analisis Geospasial: Distribusi Order & Kualitas Layanan per Negara Bagian Brazil',
              fontsize=14, fontweight='bold')

top_states = state_analysis.head(15)

# Plot 1: Volume Order per state
colors_state = sns.color_palette('Blues_r', n_colors=15)
axes[0].barh(top_states['customer_state'][::-1],
              top_states['total_orders'][::-1] / 1e3,
              color=colors_state)
axes[0].set_title('Top 15 Negara Bagian: Volume Order', fontsize=12)
axes[0].set_xlabel('Jumlah Order (Ribu)', fontsize=11)
axes[0].set_ylabel('Negara Bagian', fontsize=11)

for i, (orders, state) in enumerate(zip(top_states['total_orders'][::-1],
                                          top_states['customer_state'][::-1])):
    axes[0].text(orders/1e3 + 0.1, i, f'{orders/1e3:.1f}K', va='center', fontsize=9)

# Plot 2: Avg Delivery Days + Avg Review per state
x = np.arange(len(top_states))
width = 0.4

ax_r = axes[1]
bars_d = ax_r.bar(x - width/2, top_states['avg_delivery_days'],
                   width, label='Avg Delivery Days', color='#FF8A65', alpha=0.8)
ax_r.set_ylabel('Rata-rata Hari Pengiriman', fontsize=11)
ax_r.set_xlabel('Negara Bagian', fontsize=11)
ax_r.set_title('Avg Delivery Days & Review Score per State', fontsize=12)
ax_r.set_xticks(x)
ax_r.set_xticklabels(top_states['customer_state'], rotation=45, ha='right')

ax_r2 = ax_r.twinx()
ax_r2.plot(x + width/2, top_states['avg_review'],
            color='#1565C0', linewidth=2, marker='D', markersize=6, label='Avg Review Score')
ax_r2.set_ylabel('Rata-rata Review Score', fontsize=11, color='#1565C0')
ax_r2.tick_params(axis='y', labelcolor='#1565C0')
ax_r2.set_ylim(1, 5.5)

lines1, labels1 = ax_r.get_legend_handles_labels()
lines2, labels2 = ax_r2.get_legend_handles_labels()
ax_r.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('dashboard/viz_geospatial_state.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Visualisasi 5 (Geospatial) tersimpan')

In [ ]:
# Simpan data utama untuk dashboard Streamlit
# main data untuk dashboard
cols_dashboard = [
    'order_id', 'customer_unique_id', 'customer_state',
    'order_purchase_timestamp', 'order_yearmonth', 'order_quarter', 'order_year',
    'product_category_name_english', 'price', 'freight_value', 'payment_value',
    'review_score', 'delivery_days', 'delivery_group'
]
cols_exist = [c for c in cols_dashboard if c in delivered_df.columns]
main_data = delivered_df[cols_exist].copy()

# Konversi timestamp ke string agar mudah di-load di Streamlit
main_data['order_purchase_timestamp'] = main_data['order_purchase_timestamp'].astype(str)
main_data['delivery_group'] = main_data['delivery_group'].astype(str)

main_data.to_csv('dashboard/main_data.csv', index=False)
rfm_df.to_csv('dashboard/rfm_data.csv', index=False)
print(f'✓ main_data.csv tersimpan: {main_data.shape}')
print(f'✓ rfm_data.csv tersimpan: {rfm_df.shape}')

## Conclusion & Recommendation

### Kesimpulan

#### Kesimpulan 1 – Kategori Produk Terlaris dan Tren Penjualan

Berdasarkan analisis pada dataset Brazilian E-Commerce (2016–2018):
- **health_beauty** adalah kategori dengan **total revenue tertinggi** (>R$ 1.2 juta), diikuti oleh **watches_gifts** dan **bed_bath_table**.
- Kategori **bed_bath_table** memiliki **volume pesanan terbanyak**, menunjukkan harga rata-rata yang lebih terjangkau namun sangat diminati.
- Tren penjualan menunjukkan **pertumbuhan konsisten** sepanjang 2017–2018, dengan **lonjakan signifikan pada November 2017** (Black Friday/musim belanja akhir tahun).
- Platform Olist mengalami pertumbuhan rata-rata **~15-20% MoM** pada periode puncaknya, menunjukkan adopsi e-commerce yang pesat di Brazil.

#### Kesimpulan 2 – Hubungan Delivery Time dan Kepuasan Pelanggan

- Terdapat **korelasi negatif yang signifikan** antara lama pengiriman dan review score pelanggan.
- Pesanan yang tiba dalam **≤7 hari** mendapat rata-rata review **~4.3–4.5/5**, sedangkan yang membutuhkan **>30 hari** turun menjadi **~3.0–3.5/5**.
- Negara bagian **di luar SP dan RJ** (seperti AM, RR, AP) mengalami delivery time rata-rata **2–3x lebih lama** dibanding São Paulo, yang berkorelasi dengan review score lebih rendah.
- Kategori produk berukuran besar seperti **office_furniture** dan **home_construction** memiliki delivery time tertinggi sekaligus review score terendah.

#### Kesimpulan 3 – Segmentasi Pelanggan (RFM Analysis)

- Mayoritas pelanggan Olist berada di segmen **Lost** (>40%), mencerminkan tingginya churn rate dan rendahnya repeat purchase.
- Segmen **Champions** (pelanggan paling loyal) hanya ~5% dari total pelanggan namun berkontribusi **~20% dari total revenue**.
- Segmen **At Risk** (~10–15%) merupakan pelanggan aktif yang mulai tidak berbelanja dan perlu segera ditangani dengan program retensi.
- Fakta bahwa **>80% pelanggan hanya melakukan 1 kali transaksi** menunjukkan bahwa Olist memiliki tantangan besar dalam membangun loyalitas pelanggan.

---

### Rekomendasi (Action Items)

#### Action Item 1: Prioritaskan Promosi pada Kategori High-Value
Alokasikan budget marketing lebih besar untuk kategori **health_beauty**, **watches_gifts**, dan **sports_leisure** karena memiliki kombinasi revenue tinggi dan review score yang baik. Manfaatkan momentum **Black Friday** dengan kampanye lebih awal (Oktober–November).

#### Action Item 2: Perbaiki SLA Pengiriman untuk Wilayah Terpencil
Bangun atau optimalkan **fulfillment center regional** di wilayah dengan delivery time tinggi (Utara/Timur Laut Brazil). Targetkan pengurangan delivery time menjadi ≤14 hari untuk semua state, karena ini langsung berdampak pada peningkatan review score dan retensi pelanggan.

#### Action Item 3: Program Retensi untuk Segmen At Risk & Lost
- **Segmen "At Risk"**: Kirim email retargeting + voucher diskon eksklusif dalam 30 hari setelah pembelian terakhir.
- **Segmen "Lost"**: Kampanye win-back dengan penawaran spesial atau program referral.
- **Segmen "Champions" & "Loyal"**: Program loyalty reward (poin, cashback, early access ke produk baru) untuk mempertahankan mereka.

#### Action Item 4: Optimalkan First-Time Buyer Conversion
Karena >80% pelanggan adalah one-time buyer, buat program **"Second Purchase Incentive"** — misalnya diskon 15% untuk pembelian kedua dalam 30 hari setelah pembelian pertama — untuk meningkatkan Frequency score dan memindahkan pelanggan ke segmen Loyal.